# 🛩️ Example: Working with ImprovedB747Env

This notebook demonstrates the usage of the improved Boeing 747 environment with normalized action and observation spaces.

## 📋 Features of ImprovedB747Env:
- ✅ Normalized action and observation spaces [-1, 1]
- ✅ Improved reward function
- ✅ Optimized for reinforcement learning
- ✅ Automatic action and state clipping

## 📋 What we will do:
1. Import the required libraries
2. Configure time parameters and the reference signal
3. Create and initialize the environment
4. Execute one simulation step
5. Analyze the results

## 📚 Importing Libraries

Loading all necessary modules:

In [1]:
# Core libraries
import gymnasium as gym
import numpy as np

# Import TensorAeroSpace to register environments
import tensoraerospace

# TensorAeroSpace modules
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step

## ⚙️ Simulation Parameters Setup

Defining time parameters and creating the reference signal:

In [ ]:
# Discretization parameters
dt = 0.01  # Discretization step (seconds)

# Generate time period
tp = generate_time_period(tn=20, dt=dt)  # 20 seconds of simulation
tps = convert_tp_to_sec_tp(tp, dt=dt)    # Convert to seconds
number_time_steps = len(tp)             # Total number of time steps

# Create step reference signal for pitch angle (theta)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=10*dt, output_rad=True), 
    [1, -1]
)

print(f"📊 Simulation parameters:")
print(f"   • Simulation time: {tp[-1]:.1f} sec")
print(f"   • Discretization step: {dt} sec")
print(f"   • Number of steps: {number_time_steps}")
print(f"   • Reference signal shape: {reference_signals.shape}")

## 🚀 Creating and Initializing the Environment

Creating ImprovedB747Env with the specified parameters:

In [3]:
# Create ImprovedB747Env
# Initial state [u, w, q, theta] in SI units (m/s, m/s, rad/s, rad)
initial_state = np.array([0.0, 0.0, 0.0, 0.0])

env = gym.make(
    "ImprovedB747-v0",
    initial_state=initial_state,
    reference_signal=reference_signals,
    number_time_steps=number_time_steps,
    dt=dt,
    initial_elevator_deg=0.0,
    use_initial_action_on_first_step=True
)

# Initialize the environment
obs, info = env.reset()

print(f"✅ Environment successfully created and initialized!")
print(f"📈 Initial observation (normalized): {obs}")
print(f"🎯 Action space: {env.action_space}")
print(f"📊 Observation space: {env.observation_space}")
print(f"📐 Maximum pitch angle: ±{np.rad2deg(env.unwrapped.max_pitch_rad):.1f} deg")
print(f"📐 Maximum angular velocity: ±{np.rad2deg(env.unwrapped.max_pitch_rate_rad_s):.1f} deg/s")
print(f"📐 Maximum stabilizer deflection: ±{env.unwrapped.max_stabilizer_angle_deg:.1f} deg")

✅ Environment successfully created and initialized!
📈 Initial observation (normalized): [0. 0. 0. 0.]
🎯 Action space: Box(-1.0, 1.0, (1,), float32)
📊 Observation space: Box(-1.0, 1.0, (4,), float32)
📐 Maximum pitch angle: ±20.0 deg
📐 Maximum angular velocity: ±5.0 deg/s
📐 Maximum stabilizer deflection: ±25.0 deg


## 🎮 Executing a Simulation Step

Applying a normalized control action:

In [4]:
# Apply normalized control action
# Action in range [-1, 1]
action = np.array([0.5], dtype=np.float32)  # 50% of maximum deflection

# Execute one simulation step
obs, reward, terminated, truncated, info = env.step(action)

print(f"🎯 Control action (normalized): {action[0]:.2f}")
print(f"📊 New observation (normalized): {obs}")
print(f"   • [pitch_error_norm, q_norm, theta_norm, prev_action_norm]")
print(f"🏆 Reward: {reward:.6f}")
print(f"🔚 Terminated: {terminated}")
print(f"⏰ Truncated: {truncated}")

🎯 Control action (normalized): 0.50
📊 New observation (normalized): [0. 0. 0. 0.]
   • [pitch_error_norm, q_norm, theta_norm, prev_action_norm]
🏆 Reward: 0.000000
🔚 Terminated: False
⏰ Truncated: False


## 📊 Analyzing the Results

Examining the internal model state:

In [5]:
# Analyze current state
print(f"📈 Current step: {env.unwrapped.current_step}")
print(f"📊 Current physical state:")
print(f"   • u (longitudinal velocity): {env.unwrapped.state[0]:.3f} m/s")
print(f"   • w (vertical velocity): {env.unwrapped.state[1]:.3f} m/s")
print(f"   • q (angular velocity): {np.rad2deg(env.unwrapped.state[2]):.3f} deg/s")
print(f"   • theta (pitch angle): {np.rad2deg(env.unwrapped.state[3]):.3f} deg")

# Current reference signal
if env.unwrapped.current_step < env.unwrapped.reference_signal.shape[1]:
    ref_theta = env.unwrapped.reference_signal[0, env.unwrapped.current_step]
    print(f"🎯 Reference pitch angle: {np.rad2deg(ref_theta):.3f} deg")

📈 Current step: 1
📊 Current physical state:
   • u (longitudinal velocity): 0.000 m/s
   • w (vertical velocity): 0.000 m/s
   • q (angular velocity): 0.000 deg/s
   • theta (pitch angle): 0.000 deg
🎯 Reference pitch angle: 0.000 deg


## Visualization: Full Episode with Proportional Control

Running a complete episode and plotting the system response:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a fresh environment for the full episode
initial_state_vis = np.array([0.0, 0.0, 0.0, 0.0])
tp_vis = generate_time_period(tn=20, dt=0.01)
number_time_steps_vis = len(tp_vis)
reference_signals_vis = np.reshape(
    unit_step(degree=5, tp=tp_vis, time_step=10*dt, output_rad=True),
    [1, -1]
)

env_vis = gym.make(
    "ImprovedB747-v0",
    initial_state=initial_state_vis,
    reference_signal=reference_signals_vis,
    number_time_steps=number_time_steps_vis,
    dt=0.01,
    initial_elevator_deg=0.0,
    use_initial_action_on_first_step=True
)

obs, _ = env_vis.reset()
observations, rewards, actions = [], [], []

done = False
while not done:
    # Simple proportional control: drive pitch error (obs[0]) toward zero
    action = np.array([np.clip(-2.0 * obs[0], -1, 1)], dtype=np.float32)
    obs, reward, terminated, truncated, _ = env_vis.step(action)
    done = terminated or truncated
    observations.append(obs.copy())
    rewards.append(reward)
    actions.append(action[0])

observations = np.array(observations)
time = np.arange(len(rewards)) * 0.01

# Create professional multi-panel figure
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Boeing 747 -- Proportional Control Response', fontsize=14, fontweight='bold')

# Plot 1: Pitch tracking error
axes[0, 0].plot(time, observations[:, 0], 'b-', linewidth=1.5, label='Pitch error (norm)')
axes[0, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Target')
axes[0, 0].fill_between(time, -0.05, 0.05, alpha=0.1, color='green', label='$\\pm$5% band')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Normalized error')
axes[0, 0].set_title('Pitch Tracking Error')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: State variables
axes[0, 1].plot(time, observations[:, 1], 'g-', linewidth=1.5, label='Pitch rate (norm)')
axes[0, 1].plot(time, observations[:, 2], 'm-', linewidth=1.5, label='Pitch angle (norm)')
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Normalized value')
axes[0, 1].set_title('State Variables')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Control action (elevator)
axes[1, 0].plot(time, actions, 'r-', linewidth=1.5)
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Action (normalized)')
axes[1, 0].set_title('Control Input (Elevator)')
axes[1, 0].set_ylim(-1.1, 1.1)
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Cumulative reward
cumulative = np.cumsum(rewards)
axes[1, 1].plot(time, cumulative, 'k-', linewidth=1.5)
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Cumulative reward')
axes[1, 1].set_title(f'Cumulative Reward (total: {cumulative[-1]:.1f})')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

env_vis.close()

## 🎉 Conclusion

In this example we successfully:

✅ **Created ImprovedB747Env** with normalized spaces
✅ **Configured the parameters** of the simulation
✅ **Executed a step** with a normalized action
✅ **Analyzed the results**

### 📝 Features of Improved environments:
- **Normalized actions**: Range [-1, 1]
- **Normalized observations**: Range [-1, 1]
- **Improved reward**: More stable training
- **Automatic constraints**: Physical limits are applied automatically

The model is ready for reinforcement learning! 🚀